# BaSSSh-seq — Bacterial scRNA-seq analysis
BaSSSh-seq pipeline — complete script
Steps: load → merge → fix cell names → HVG/PCA/BBKNN → UMAP/Leiden → markers → annotation → save
Step 7 (CellTypist training) is NOT included.

**Order:** Import → Preprocessing (run once) → Load → Merge → HVG/PCA/BBKNN → UMAP/Leiden → Marker genes → Annotation → CellTypist

## 1. Imports

In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import celltypist

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

/home/nvanacker/miniconda3/envs/inab/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/nvanacker/miniconda3/envs/inab/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/nvanacker/miniconda3/envs/inab/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/nvanacker/miniconda3/envs/inab/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/home/nvanacker/miniconda3/envs/inab/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarn

In [2]:
print("=== STEP 1: Load datasets ===")
#adata_bf = sc.read_h5ad("../data/BF_gefilterd.h5ad", backed="r")
#adata_p  = sc.read_h5ad("../data/P_gefilterd.h5ad",  backed="r")


 

=== STEP 1: Load datasets ===


## 2a. BF preprocessing — Run once, then comment out
This cell reads the raw BF count matrix, removes rRNA genes, filters cells on ≥ 7 non-rRNA reads, normalises, and saves the result as BF_gefilterd.h5ad

In [ ]:
# ============================================================
# Run once → comment out entirely afterwards
# ============================================================

#adata_bf = sc.read_h5ad("../data/BF_count_mat.h5ad")
#adata_bf.var_names_make_unique()

# # rRNA removal (paper: all rRNA genes removed before analysis)
# # Bacterial rRNA: 5S, 16S, 23S
#adata_bf.var["rrna"] = (
     #adata_bf.var_names.str.lower().str.startswith(("rrs", "rrl", "rrf")) |
     #adata_bf.var_names.str.lower().str.contains("rrna") |
     #adata_bf.var_names.str.lower().str.contains("16s") |
     #adata_bf.var_names.str.lower().str.contains("23s") |
     #adata_bf.var_names.str.lower().str.contains("5s_rrna")
#)
#print(f"rRNA genes found (BF): {adata_bf.var['rrna'].sum()}")
#adata_bf = adata_bf[:, ~adata_bf.var["rrna"]].copy()

# # Count filter: paper ≥ 7 non-rRNA reads for BF → expected ~3680 cells
#adata_bf.obs["total_counts"] = adata_bf.X.sum(axis=1)
#print(f"Cells before filter: {adata_bf.n_obs}")
#adata_bf = adata_bf[adata_bf.obs["total_counts"] >= 7].copy()
#print(f"Cells after filter (≥7): {adata_bf.n_obs}")  # Paper: 3680 cells

# # Normalisation: 10^4 counts, log+1 transform (paper Methods)
#sc.pp.normalize_total(adata_bf, target_sum=1e4)
#sc.pp.log1p(adata_bf)

# # Label for merge
#adata_bf.obs["sample"] = "bf"

# # Save
#adata_bf.write("../data/BF_gefilterd.h5ad", compression="gzip")
#print("BF saved!")

## 2b. Load BF
After filtering on ≥7 non-rRNA reads per cell, we obtained 3,668 biofilm cells compared to 3,680 reported in the paper — a difference of 12 cells (~0.33%), likely due to minor package version differences and considered negligible.

In [3]:
# Load the preprocessed BF data (memory-mapped, low RAM)
adata_bf = sc.read_h5ad("../data/BF_gefilterd.h5ad", backed="r")
print(f"BF: {adata_bf.shape[0]} cells, {adata_bf.shape[1]} genes")

BF: 3668 cells, 2853 genes


## 3a. Planktonic preprocessing — Run once, then comment out
This cell reads the raw Planktonic count matrix, removes rRNA genes, filters cells on ≥ 28 non-rRNA reads, normalises, and saves the result as P_gefilterd.h5ad.

In [ ]:
# ============================================================
# Run once → comment out entirely afterwards
# ============================================================

#adata_p = sc.read_h5ad("../data/P_count_mat.h5ad")
#adata_p.var_names_make_unique()

# # Same rRNA removal as BF
#adata_p.var["rrna"] = (
     #adata_p.var_names.str.lower().str.startswith(("rrs", "rrl", "rrf")) |
     #adata_p.var_names.str.lower().str.contains("rrna") |
     #adata_p.var_names.str.lower().str.contains("16s") |
     #adata_p.var_names.str.lower().str.contains("23s") |
    # adata_p.var_names.str.lower().str.contains("5s_rrna")
#)
#print(f"rRNA genes found (P): {adata_p.var['rrna'].sum()}")
#adata_p = adata_p[:, ~adata_p.var["rrna"]].copy()

# # Count filter: paper ≥ 28 non-rRNA reads for Planktonic → expected ~4231 cells
#adata_p.obs["total_counts"] = adata_p.X.sum(axis=1)
#print(f"Cells before filter: {adata_p.n_obs}")
#adata_p = adata_p[adata_p.obs["total_counts"] >= 28].copy()
#print(f"Cells after filter (≥28): {adata_p.n_obs}")  # Paper: 4231 cells

# # Normalisation
#sc.pp.normalize_total(adata_p, target_sum=1e4)
#sc.pp.log1p(adata_p)

# # Label for merge
#adata_p.obs["sample"] = "p"

# # Save
#adata_p.write("../data/P_gefilterd.h5ad", compression="gzip")
#print("Planktonic saved!")

## 3b. Load Planktonic
After filtering on ≥28 non-rRNA reads per cell, we obtained 4,216 planktonic cells compared to 4,231 reported in the paper — a difference of 15 cells (~0.35%), likely due to minor package version differences and considered negligible.

In [4]:
# Load the preprocessed Planktonic data (memory-mapped, low RAM)
adata_p = sc.read_h5ad("../data/P_gefilterd.h5ad", backed="r")
print(f"P:  {adata_p.shape[0]} cells,  {adata_p.shape[1]} genes")


P:  4216 cells,  2853 genes


## 4. Merge BF + Planktonic
Paper: cells from both conditions combined for BBKNN-integrated clustering

In [5]:
adata_all_genes = ad.concat(
    {"bf": adata_bf, "p": adata_p},
    label="sample",
    join="inner"
)

In [6]:
#strip de bestaande prefix die ad.concat al toevoegt, dan underscore versie maken
adata_all_genes.obs_names = [
    barcode.replace("bf-", "bf_").replace("p-", "p_")
    for barcode in adata_all_genes.obs_names
]

# Verify
print(f"Cell names: {list(adata_all_genes.obs_names[:3])}")
print(f"Shape: {adata_all_genes.shape}")

Cell names: ['68', '96', '104']
Shape: (7884, 2853)


In [7]:
# Reset de sample kolom zodat categorieën uniek zijn
adata_all_genes.obs["sample"] = adata_all_genes.obs["sample"].astype(str)

save_path = "/mnt/c/Users/natha/OneDrive/Bio-informatica_25-26/International Internship/Internship-25-26/CellTypist/celltypist_pipeline/data/bacteria_all_genes_combined.h5ad"

adata_all_genes.write(save_path, compression="gzip")
print("All-genes dataset saved!")

... storing 'sample' as categorical


All-genes dataset saved!


In [ ]:
# =============================================================
# STEP 5: HVG → Scale → PCA → BBKNN
# =============================================================
print("\n=== STEP 5: HVG + PCA + BBKNN ===")
 
# Store raw counts before filtering (required for marker genes & CellTypist)
adata_bacteria.raw = adata_bacteria
 
sc.pp.highly_variable_genes(
    adata_bacteria,
    min_mean=0.00625,
    min_disp=0.25,
    batch_key='sample'
)
n_hvg = adata_bacteria.var.highly_variable.sum()
print(f"Number of highly variable genes: {n_hvg}  (expected ~1015)")
 
# Filter to HVGs
adata_bacteria = adata_bacteria[:, adata_bacteria.var.highly_variable].copy()
 
# Scale & PCA
sc.pp.scale(adata_bacteria, max_value=10)
sc.tl.pca(adata_bacteria, n_comps=50)
 
# BBKNN batch correction (paper: neighbors_within_batch=9, n_pcs=4)
sc.external.pp.bbknn(
    adata_bacteria,
    batch_key='sample',
    neighbors_within_batch=9,
    n_pcs=4
)
print("HVG + PCA + BBKNN done!")

In [ ]:
adata_bacteria.obs_names = adata_bacteria.obs_names.str.replace("-", "_")
adata_bacteria.write("../data/bacteria_combined_annotated.h5ad", compression="gzip")

# Verify
test = sc.read_h5ad("../data/bacteria_combined_annotated.h5ad")
print(f"Shape: {test.shape}")
print(f"Columns: {test.obs.columns.tolist()}")
print(f"Cell names: {list(test.obs_names[:3])}")

In [ ]:
#1. fix the cells names, remove the -1.
#2. takes into account of -1, if I have duplicated cell names (check this)
#3. be sure that the new adata object have the changes, and the same highly variable genes as the celltypist?

In [ ]:
#pre adata object in order to be sure that I have the correct number of cells and genes. I expect 78.. cells and 1015

In [ ]:
#here I have to save it as a .h5ad file, because you work with this. This is not comparable


In [ ]:
#graphical embedding: 
#check correlation of cells in teach neigho

#MDS 
#takes one cell anc check the correlation ofhe whole cells 